# 02. Как строится гибридный RAG и генерируется ответ

**Цель:** проследить один вопрос от двух поисковых выдач до ответа со ссылками.
Нужен индекс из [01](01_hybrid_rag.ipynb), но его kernel можно закрыть.

```text
вопрос ┬→ dense ─┐
       └→ BM25 ──┴→ RRF → reranker → top-k → соседи → контекст → LLM → ответ
```

Здесь алгоритмы находятся в ячейках. В helpers остаются HTTP-клиенты, настройки,
идентификаторы и сохранение файлов. Эталонные ответы загружаются для последующей
оценки, но никогда не передаются в поиск или генерацию.
**Run All вызывает модели, выполняет сравнения и пакетный inference.**
Для одной пробы остановитесь перед разделом 8. [Настройка сервисов](SETUP.md).

## 1. Откроем готовый индекс

Используем мультимодальный индекс из 01: текст и описания картинок ищутся вместе.
VLM повторно не вызывается. Наследуем параметры индекса, чтобы не перепутать embedding-модель,
шаблон запроса и коллекцию. Настройки поиска и генерации можно менять отдельно.
После изменения их или промпта выберите новый `RUN_NAME`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "helpers" / "artifacts.py").exists():
    ROOT = ROOT / "hybrid_rag_notebooks"
if not (ROOT / "helpers" / "artifacts.py").exists():
    raise RuntimeError("Запустите Jupyter из hybrid_rag_notebooks или её родительской папки")
sys.path.insert(0, str(ROOT))
from helpers.settings import prepare_environment  # noqa: E402

prepare_environment(ROOT)

In [ ]:
import json
import time
from copy import deepcopy

import pandas as pd
from fastembed import SparseTextEmbedding
from helpers.artifacts import (
    append_jsonl,
    ensure_manifest,
    fingerprint,
    latest_records,
    load_questions,
)
from helpers.model_clients import ModelAPI
from helpers.retrieval import chunk_text, point_id, prepare_collection, sparse_vector
from helpers.settings import (
    ATTENTION_ANSWER_PROMPT,
    model_server_commands,
    rag_settings,
)
from qdrant_client import QdrantClient
from qdrant_client import models as qm
from tqdm.auto import tqdm

CFG = rag_settings()
CFG.update(vision=True, translate=False)
INDEX_NAME = "attention_multimodal_v1"
RUN_NAME = "attention_multimodal_hybrid_v1"
INDEX_DIR = ROOT / "outputs" / "indexes" / INDEX_NAME
RUN_DIR = ROOT / "outputs" / "runs" / RUN_NAME
QUESTIONS_PATH = ROOT / "data" / "questions.jsonl"
SEARCH = dict(
    mode="hybrid",
    k=8,
    candidates=30,
    neighbors=2,
    context_tokens=20000,
    rerank=True,
    translate=False,
    show_branches=True,
)
ANSWER_PROMPT = ATTENTION_ANSWER_PROMPT
print(ANSWER_PROMPT)

In [ ]:
manifest_path = INDEX_DIR / "index.json"
if not manifest_path.exists():
    raise FileNotFoundError("Выполните ноутбук 01 или выберите существующий INDEX_NAME")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
INDEX_CONFIG = manifest["config"]
if manifest["fingerprint"] != fingerprint(INDEX_CONFIG):
    raise ValueError("Повреждён манифест индекса")
if not (INDEX_DIR / "binding.json").exists():
    raise FileNotFoundError("Отсутствует binding.json; завершите индексирование в ноутбуке 01")
ensure_manifest(INDEX_DIR / "binding.json", INDEX_CONFIG)
# Параметры embeddings/BM25 и Qdrant наследуются из индекса, а не из текущего .env.
CFG.update({key: value for key, value in INDEX_CONFIG.items() if key in CFG})
COLLECTION = INDEX_CONFIG["collection"]
dimension = INDEX_CONFIG["dimension"]
print("Индекс:", INDEX_NAME, "коллекция:", COLLECTION, "модель:", CFG["embed_model"])
# Описания картинок уже в индексе: на этапе поиска VLM повторно не вызывается.
for role, command in model_server_commands({**CFG, "vision": False}).items():
    print(role, command, sep="\n")

In [ ]:
embedder = ModelAPI(CFG["embed_url"], CFG["embed_model"], "EMBED_API_KEY")
generator = ModelAPI(CFG["llm_url"], CFG["llm_model"], "LLM_API_KEY")
reranker = (
    ModelAPI(CFG["rerank_url"], CFG["rerank_model"], "RERANK_API_KEY") if SEARCH["rerank"] else None
)
for name, client in [("embed", embedder), ("llm", generator), ("rerank", reranker)]:
    if client is not None:
        available = [m["id"] for m in client.models()["data"]]
        if client.model not in available:
            raise ValueError(f"{name}: {client.model} отсутствует в {available}")
qdrant = QdrantClient(url=CFG["qdrant_url"], timeout=120)
if not qdrant.collection_exists(COLLECTION):
    raise ValueError("Коллекция отсутствует: восстановите индекс в ноутбуке 01")
prepare_collection(qdrant, COLLECTION, dimension)
expected_count = sum(
    len(json.loads(p.read_text(encoding="utf-8"))) for p in (INDEX_DIR / "parsed").glob("*.json")
)
if not expected_count or qdrant.count(COLLECTION, exact=True).count != expected_count:
    raise ValueError("Индекс неполный: повторите индексирование в ноутбуке 01")
print("Загрузка BM25…", flush=True)
bm25 = SparseTextEmbedding(CFG["sparse_model"])
print("Готово, чанков:", expected_count)

## 2. Две ветви поиска

Dense ищет смысловую близость вопроса и фрагмента, BM25 — совпадения терминов.
Сырые оценки у них разных шкал, поэтому объединяем **позиции**, а не складываем cosine и BM25.
Функция ниже делает явные `query_points`. Она вызывается и для одиночного вопроса,
и в пакетном inference — используется один алгоритм.

С каждой ветви берём `4 × max(k, candidates)` результатов; RRF оставляет
`max(k, candidates)` кандидатов. Этот пул одинаков при сравнении с reranker и без него.

In [ ]:
def retrieve_candidates(question, settings):
    mode = settings["mode"]
    if mode not in ("dense", "bm25", "hybrid"):
        raise ValueError("mode: dense, bm25 или hybrid")
    queries = [question]
    if settings["translate"]:
        translated = generator.chat(
            [
                {
                    "role": "user",
                    "content": "Translate the question to English for document retrieval. Return only the translation.\n"
                    + question,
                }
            ],
            temperature=0,
        )
        text = translated["choices"][0]["message"]["content"].strip()
        if text and text.casefold() != question.casefold():
            queries.append(text)
    count = max(settings["k"], settings["candidates"])
    prefetch, branches = [], []
    for query in queries:
        vectors = {}
        if mode in ("dense", "hybrid"):
            vectors["dense"] = embedder.embed(
                [CFG["query_template"].format(query=query)], dim=dimension
            )[0]
        if mode in ("bm25", "hybrid"):
            vectors["bm25"] = sparse_vector(next(iter(bm25.query_embed(query))))
        for name, vector in vectors.items():
            hits = qdrant.query_points(
                COLLECTION, query=vector, using=name, limit=count * 4, with_payload=True
            ).points
            branches.extend(
                {
                    "query": query,
                    "branch": name,
                    "rank": i + 1,
                    "chunk_id": h.payload["chunk_id"],
                    "score": h.score,
                    "text": h.payload["text"],
                }
                for i, h in enumerate(hits)
            )
            prefetch.append(qm.Prefetch(query=vector, using=name, limit=count * 4))
    fused = qdrant.query_points(
        COLLECTION,
        prefetch=prefetch,
        query=qm.FusionQuery(fusion=qm.Fusion.RRF),
        limit=count,
        with_payload=True,
    ).points
    candidates = [
        {
            "payload": h.payload,
            "rrf_rank": i + 1,
            "rrf_score": h.score,
            "score": h.score,
        }
        for i, h in enumerate(fused)
    ]
    return candidates, branches, queries

In [ ]:
questions = load_questions(QUESTIONS_PATH)
QUESTION_ID = "attn_08_head_dimensions"
CUSTOM_QUESTION = ""
question = CUSTOM_QUESTION.strip() or next(
    q["question"] for q in questions if q["id"] == QUESTION_ID
)
print("Вопрос:", question)
print("Поиск:", SEARCH)

In [ ]:
search_started = time.perf_counter()
candidates, branches, queries = retrieve_candidates(question, SEARCH)
for name in ("dense", "bm25"):
    table = pd.DataFrame([r for r in branches if r["branch"] == name])
    print("\nВетвь:", name)
    print(table.head(5).to_string(index=False))

## 3. RRF: объединение по позициям

В используемом `FusionQuery(Fusion.RRF)` Qdrant применяет вклад
`1 / (2 + rank - 1)`, где `rank` начинается с 1. Это параметр RRF,
а не число кандидатов. Проверяем формулу на реальных списках предыдущей ячейки,
затем сравниваем её с возвращённой Qdrant оценкой.
При равных суммах порядок может отличаться; эти равенства не доказывают различий качества.

In [ ]:
rrf = {}
for row in branches:
    key = row["chunk_id"]
    rrf[key] = rrf.get(key, 0.0) + 1 / (2 + row["rank"] - 1)
fusion_table = pd.DataFrame(
    [
        {
            "chunk_id": h["payload"]["chunk_id"],
            "rrf_rank": h["rrf_rank"],
            "qdrant_rrf": h["rrf_score"],
            "manual_rrf": rrf[h["payload"]["chunk_id"]],
            "text": h["payload"]["text"][:300],
        }
        for h in candidates
    ]
)
print(fusion_table.head(10).to_string(index=False))

## 4. Reranker и соседние чанки

Reranker оценивает пару «вопрос, найденный текст» и меняет порядок небольшого
пула кандидатов. Он не ищет новые документы. В HTTP-клиенте длинный текст
ограничивается 4000 символами — это может повлиять на оценку больших чанков.
При ошибке здесь останавливаемся: исправьте сервис или явно выключите reranker.

In [ ]:
def rerank_candidates(question, candidates, enabled):
    ranked = deepcopy(candidates)
    if enabled and ranked:
        if reranker is None:
            raise RuntimeError("Подключите reranker или выключите SEARCH['rerank']")
        scores = reranker.rerank(question, [chunk_text(h["payload"]) for h in ranked])
        for hit, score in zip(ranked, scores, strict=True):
            hit.update(score=score, rerank_score=score)
        ranked.sort(key=lambda h: -h["score"])
    return ranked

In [ ]:
ranked = rerank_candidates(question, candidates, SEARCH["rerank"])
print(
    pd.DataFrame(
        [
            {
                "chunk_id": h["payload"]["chunk_id"],
                "rrf_rank": h["rrf_rank"],
                "new_rank": i + 1,
                "rerank_score": h.get("rerank_score"),
                "text": h["payload"]["text"][:200],
            }
            for i, h in enumerate(ranked)
        ]
    )
    .head(10)
    .to_string(index=False)
)
hits = ranked[: SEARCH["k"]]

Теперь добавим до двух соседей с каждой стороны, только внутри той же секции.
Одинаковые чанки не должны попадать в контекст дважды. Границу секции проверяем
по `headings`; если метаданные пустые, это ограничение менее надёжно.

In [ ]:
def expand_neighbors(hits, n):
    known = {h["payload"]["chunk_id"]: h["payload"] for h in hits}
    wanted = set()
    for hit in hits:
        doc_id, index = hit["payload"]["chunk_id"].rsplit(":", 1)
        wanted.update(
            f"{doc_id}:{i:05d}" for i in range(max(0, int(index) - n), int(index) + n + 1)
        )
    if wanted:
        for point in qdrant.retrieve(
            COLLECTION, ids=[point_id(k) for k in sorted(wanted)], with_payload=True
        ):
            known[point.payload["chunk_id"]] = point.payload
    covered, items = set(), []
    for hit in hits:
        payload = hit["payload"]
        key = payload["chunk_id"]
        if key in covered:
            continue
        doc_id, index = key.rsplit(":", 1)
        block = [key]
        for step in (-1, 1):
            for i in range(int(index) + step, int(index) + step * (n + 1), step):
                neighbor = f"{doc_id}:{i:05d}"
                other = known.get(neighbor)
                if (
                    i < 0
                    or other is None
                    or neighbor in covered
                    or (other.get("headings") or []) != (payload.get("headings") or [])
                ):
                    break
                block = [neighbor, *block] if step < 0 else [*block, neighbor]
        covered.update(block)
        first = known[block[0]]
        headings = first.get("headings") or []
        text = (" > ".join(headings) + "\n" if headings else "") + "\n".join(
            known[k]["text"] for k in block
        )
        pages = [known[k]["page"] for k in block if known[k].get("page") is not None]
        items.append(
            {
                "ref": key,
                "chunk_ids": block,
                "text": text,
                "path": first.get("path"),
                "page": min(pages) if pages else None,
                "score": hit["score"],
            }
        )
    return items

In [ ]:
items = expand_neighbors(hits, SEARCH["neighbors"])
print(
    pd.DataFrame(
        [{"anchor": x["ref"], "included": x["chunk_ids"], "page": x["page"]} for x in items]
    ).to_string(index=False)
)

## 5. Точный контекст для генератора

Нумеруем источники `[1]`, `[2]` — это номера блоков, не страницы PDF.
Оценка бюджета здесь приблизительная: `символы / 4`, без вопроса, инструкции и ответа.
Первый блок сохраняется даже сверх бюджета, чтобы не получить пустой контекст.
Поэтому проверяйте размер контекстного окна модели; это не строгий токеновый ограничитель.

In [ ]:
def assemble_context(items, max_tokens):
    blocks, sources, used = [], [], 0
    for item in items:
        cost = max(1, len(item["text"]) // 4)
        if sources and used + cost > max_tokens:
            break
        n = len(sources) + 1
        label = f"document {item['path'] or item['ref']}"
        if item.get("page") is not None:
            label += f", p.{item['page']}"
        block = f"[{n}] ({label})\n{item['text']}"
        blocks.append(block)
        sources.append({**item, "n": n, "context_block": block})
        used += cost
    return "\n\n".join(blocks), sources, used

In [ ]:
context, sources, used = assemble_context(items, SEARCH["context_tokens"])
found = {
    "question": question,
    "queries": queries,
    "candidates": ranked,
    "branches": branches,
    "sources": sources,
    "retrieved_contexts": [s["context_block"] for s in sources],
    "context": context,
    "context_tokens_estimate": used,
    "latency_ms": round((time.perf_counter() - search_started) * 1000),
}
print(found["context"] or "Источники не найдены")
print("Приблизительно токенов:", used)

## 6. Inference: контекст + вопрос → ответ

Системная инструкция и пользовательский запрос видны ниже полностью.
Текст источников — доказательство, а не инструкция. Генерация использует `found`,
**не выполняя повторного поиска**. RAG — это поиск и подготовка входа для LLM;
обучения весов модели здесь нет.

In [ ]:
SYSTEM_PROMPT = "You answer questions from supplied sources. Follow the citation and evidence rules in the task."


def generate_answer(found, answer_prompt=None):
    if answer_prompt is None:
        answer_prompt = ANSWER_PROMPT
    started = time.perf_counter()
    usage = {}
    response = "В базе знаний нет релевантных источников для этого вопроса."
    if found["sources"]:
        prompt = answer_prompt.format(question=found["question"], context=found["context"])
        completion = generator.chat(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            temperature=0.1,
        )
        response = completion["choices"][0]["message"]["content"]
        if not response or not response.strip():
            raise ValueError("Генератор вернул пустой ответ")
        usage = completion.get("usage", {})
    return {
        **found,
        "response": response,
        "usage": usage,
        "latency_ms": found["latency_ms"] + round((time.perf_counter() - started) * 1000),
    }

In [ ]:
print("SYSTEM:\n", SYSTEM_PROMPT)
print(
    "USER:\n",
    ANSWER_PROMPT.format(question=found["question"], context=found["context"]),
)

In [ ]:
answer = generate_answer(found)
print(answer["response"])
print("Поиск + генерация, мс:", answer["latency_ms"], "usage:", answer["usage"])

## 7. Сравним варианты и проверим отказ

Соберём уже изученные шаги в функцию для повторных экспериментов.
Один и тот же путь используется для таблицы сравнения и пакета ответов.
Время пошагового примера выше включает паузы между ячейками; автоматический
путь ниже измеряет работу без этих пауз.

In [ ]:
def search_question(question, settings):
    question = question.strip()
    if not question:
        raise ValueError("Введите непустой вопрос")
    started = time.perf_counter()
    candidates, branches, queries = retrieve_candidates(question, settings)
    ranked = rerank_candidates(question, candidates, settings["rerank"])
    items = expand_neighbors(ranked[: settings["k"]], settings["neighbors"])
    context, sources, used = assemble_context(items, settings["context_tokens"])
    return {
        "question": question,
        "queries": queries,
        "candidates": ranked,
        "branches": branches,
        "sources": sources,
        "retrieved_contexts": [s["context_block"] for s in sources],
        "context": context,
        "context_tokens_estimate": used,
        "latency_ms": round((time.perf_counter() - started) * 1000),
    }

In [ ]:
comparison = []
variants = [("dense", False), ("bm25", False), ("hybrid", False)]
if SEARCH["rerank"] and reranker is not None:
    variants.append(("hybrid", True))
else:
    print("Сравнение с reranker пропущено: он выключен")
for mode, use_reranker in tqdm(
    variants,
    desc="Сравнение",
    unit="вариант",
):
    result = search_question(question, {**SEARCH, "mode": mode, "rerank": use_reranker})
    response = generate_answer(result)
    comparison.append(
        {
            "mode": mode,
            "rerank": use_reranker,
            "sources": [s["ref"] for s in result["sources"]],
            "answer": response["response"],
            "ms": response["latency_ms"],
        }
    )
print(pd.DataFrame(comparison).to_string(index=False))

In [ ]:
refusal_question = next(
    (q["question"] for q in questions if not q.get("reference")),
    "Какой random seed использован в статье?",
)
refusal_found = search_question(refusal_question, SEARCH)
refusal_answer = generate_answer(refusal_found)
print(refusal_question)
print(refusal_found["context"])
print("Ответ:", refusal_answer["response"])
# Ожидаем признание отсутствия данных; наличие найденных чанков не доказывает наличие ответа.

Проверьте вручную: есть ли нужный факт в источниках, корректны ли числа и ссылки?
На одном вопросе нельзя доказать, что hybrid всегда лучше. Сравнение на полном
наборе выполняется в третьем ноутбуке.

**Дополнительный эксперимент:** русские вопросы к английской статье могут
выигрывать от перевода. Следующая ячейка по умолчанию его не вызывает.
При включении появляется отдельный запрос к генератору; его выдача — только поисковая формулировка.

In [ ]:
TRY_TRANSLATION = False
if TRY_TRANSLATION:
    translated_found = search_question(question, {**SEARCH, "translate": True})
    print("Формулировки:", translated_found["queries"])
    print(translated_found["context"])

## 8. Пакет ответов для Ragas

Эта ячейка обрабатывает **весь набор**. Успешные вопросы пропускаются,
ошибки повторяются; результат записывается после каждого вопроса.
Для другой конфигурации выберите новый `RUN_NAME` в настройках и повторите ячейки.
Один каталог должен иметь только одного писателя.

In [ ]:
RUN_DIR = ROOT / "outputs" / "runs" / RUN_NAME
run_config = {
    "index": INDEX_CONFIG,
    "retrieval": {**CFG, **SEARCH},
    "answer_prompt": ANSWER_PROMPT,
    "system_prompt": SYSTEM_PROMPT,
    "questions": questions,
    "implementation": 3,
}
ensure_manifest(RUN_DIR / "run.json", run_config)
previous = latest_records(RUN_DIR / "answers.jsonl")
pending = [q for q in questions if previous.get(q["id"], {}).get("status") != "ok"]
print(
    f"Вопросов: {len(questions)}; готово: {len(questions) - len(pending)}; осталось: {len(pending)}",
    flush=True,
)
for sample in tqdm(pending, desc="Inference", unit="вопрос"):
    try:
        # Только вопрос и контекст попадают в модель. Эталон добавляем после генерации.
        result = generate_answer(search_question(sample["question"], SEARCH), ANSWER_PROMPT)
        result.pop("candidates")
        result.pop("branches")
        row = {**sample, **result, "status": "ok"}
    except Exception as exc:
        row = {**sample, "status": "error", "error": f"{type(exc).__name__}: {exc}"}
        tqdm.write(f"{sample['id']}: {row['error']}")
    append_jsonl(RUN_DIR / "answers.jsonl", row)
answers = latest_records(RUN_DIR / "answers.jsonl")
failed = [q["id"] for q in questions if answers.get(q["id"], {}).get("status") != "ok"]
print("Сохранено:", len(questions) - len(failed), "ошибок/пропусков:", len(failed))
if failed:
    raise RuntimeError(f"Исправьте ошибки и повторите ячейку: {failed}")

In [ ]:
saved = answers[questions[0]["id"]]
print(
    json.dumps(
        {
            key: saved.get(key)
            for key in (
                "id",
                "question",
                "response",
                "retrieved_contexts",
                "reference",
                "status",
            )
        },
        ensure_ascii=False,
        indent=2,
    )
)
print("Для Ragas: RUN_NAME =", repr(RUN_NAME))

Откройте [03 — Ragas](03_hybrid_rag_ragas.ipynb) с тем же `RUN_NAME`.
Для сравнения сохраните ещё один пакет: например, новый `RUN_NAME`, `mode="dense"`,
`rerank=False`, остальные параметры прежние. Повторите пакетную ячейку.
Существующие `RAGLab` и `HybridRetriever` оставлены для совместимости со старым
маршрутом, а этот ноутбук показывает алгоритм напрямую.

In [ ]:
for client in (embedder, generator, reranker):
    if client is not None:
        client.close()
qdrant.close()